# Evo 2 on the crosstalk genomic rung

**What this is.** Sections 14, 17 and 18 of `crosstalk/FINDINGS.md` establish that a
genomic language model scoring a native coding sequence carries no protein-fitness
signal: mean Spearman **-0.013** across 25 DMS assays where ESM-2 650M reaches
**+0.466**, and chance-level AUC on the ParD3 binding-specificity task across four
Nucleotide Transformer scales and HyenaDNA.

Every one of those models is small and none is Evo 2. Evo 2 is the strongest
available genomic LM and the obvious objection to the result, so it is the one
arm that has to run on a GPU. This notebook runs it under the **identical**
protocol and writes out numbers that drop straight into the existing tables.

**What you need.** An **A100 or L4** runtime (Runtime -> Change runtime type). Evo 2
uses Transformer Engine, which needs compute capability >= 8.0; a T4 (7.5) will
not work. Runtime is roughly 30-90 minutes for the default settings.

**What to send back.** The final cell writes `evo2_crosstalk_output.zip`. That file
is the whole deliverable.

Cells are independent and each writes its results to disk as it goes, so a crash
part way through does not lose the earlier arms.

## 1. Check the GPU before installing anything

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> A100 or L4.")
cap = torch.cuda.get_device_capability()
print("compute capability", cap)
if cap[0] < 8:
    raise SystemExit(f"Compute capability {cap} is too low for Evo 2 "
                     "(needs >= 8.0). Use an A100 or L4, not a T4.")
print("OK")

In [ ]:
import os, pathlib
# Output directory. On Colab this was /content; on a cluster set OUTDIR to
# somewhere on scratch. Every write in this notebook goes through it.
OUTDIR = os.environ.get("OUTDIR", "/content" if os.path.isdir("/content") else ".")
pathlib.Path(OUTDIR).mkdir(parents=True, exist_ok=True)

# Model. evo2_1b_base requires FP8 via Transformer Engine; where TE is absent it
# raises ImportError and the remedy is a 7b-class model. 7B is also the stronger
# test of "the strongest available genomic LM", so it is the better default, but
# it changes which model the reported numbers describe. Record the choice.
MODEL = os.environ.get("EVO2_MODEL", "evo2_7b")
print("OUTDIR =", OUTDIR, "| MODEL =", MODEL)

## 2. Install Evo 2

The install is the fragile part. PyPI first, then a source build, and the cell
reports which route worked so a failure here is unambiguous.

In [ ]:
import importlib, subprocess, sys

def sh(cmd):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True).returncode

route = None
try:
    import evo2; route = "already installed"
except ImportError:
    if sh("pip install -q evo2") == 0:
        try:
            importlib.invalidate_caches(); import evo2; route = "pypi"
        except ImportError: pass
    if route is None:
        sh(f"git clone -q https://github.com/ArcInstitute/evo2.git {OUTDIR}/evo2src")
        sh(f"pip install -q {OUTDIR}/evo2src")
        importlib.invalidate_caches()
        import evo2; route = "source"
print("Evo 2 available via:", route)

## 3. Load the model and smoke-test the scorer

`MODEL` is the only knob most people need. `evo2_1b_base` fits comfortably on an
L4; `evo2_7b` needs an A100 40GB. The scoring helper is written against the raw
logits rather than any convenience method, so it behaves the same whichever
version of the package installed, and it returns a **sum** of token log-probs to
match the autoregressive likelihood used for HyenaDNA in section 14.

In [ ]:
BATCH  = 8

import numpy as np, torch
from evo2 import Evo2

model = Evo2(MODEL)
print("loaded", MODEL)

@torch.inference_mode()
def evo2_loglik(seqs, batch=BATCH, progress=2000):
    """Sum of autoregressive token log-probabilities, one value per sequence."""
    out = []
    for i in range(0, len(seqs), batch):
        chunk = seqs[i:i+batch]
        toks = [torch.tensor(model.tokenizer.tokenize(s), dtype=torch.long) for s in chunk]
        n = max(len(t) for t in toks)
        pad = model.tokenizer.pad_id if hasattr(model.tokenizer, "pad_id") else 0
        ids = torch.full((len(toks), n), pad, dtype=torch.long)
        for j, t in enumerate(toks):
            ids[j, :len(t)] = t
        ids = ids.cuda()
        logits = model(ids)
        # evo2 0.6.0 nests two deep: ((logits, inference_params), None). One
        # unwrap is a level short and the next line raises TypeError. Unwrap
        # until it is a tensor rather than assuming a depth.
        while isinstance(logits, (tuple, list)):
            logits = logits[0]
        lp = torch.log_softmax(logits[:, :-1].float(), -1)
        tgt = ids[:, 1:]
        tok_lp = lp.gather(-1, tgt.unsqueeze(-1)).squeeze(-1)
        for j, t in enumerate(toks):
            out.append(float(tok_lp[j, :len(t)-1].sum()))
        if progress and (i + batch) % progress < batch:
            print(f"   {min(i+batch, len(seqs))}/{len(seqs)}", flush=True)
    return np.array(out)

s = evo2_loglik(["ATGGCAAACGTGGAAAAAATGAGCGTGGCC", "TTTTTTTTTTTTTTTTTTTTTTTTTTTTTT"])
print("smoke test:", np.round(s, 2), "-- the coding-like sequence should score higher")
assert np.isfinite(s).all(), "scorer returned non-finite values"

## 4. The ParD3 landscape, embedded

The measured landscape (Lite et al. 2020, GEO GSE153897), the verified native CDS,
the real ParD3:ParE3 operon and an 11 kb window of the *M. opportunistum*
chromosome around the locus, gzipped into the notebook so nothing needs
downloading and the numbers match the local run exactly.

In [ ]:
import base64, gzip, json
BLOB = "..."
D = json.loads(gzip.decompress(base64.b64decode(BLOB)))
variants = D["variants"]; w3 = np.array(D["w3"]); w2 = np.array(D["w2"])
wt_cds = D["wt_cds"]; PARD3 = D["pard3"]; MUT_POSITIONS = D["mut_positions"]
print(len(variants), "variants;", len(wt_cds), "nt CDS")

BASES="TCAG"; AAS="FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG"
CODON={b1+b2+b3:AAS[i] for i,(b1,b2,b3) in enumerate(
       (a,b,c) for a in BASES for b in BASES for c in BASES)}
SYN={}
for c,a in CODON.items(): SYN.setdefault(a,[]).append(c)
usage=D["codon_usage"]
PREF={a:max(cs,key=lambda c: usage.get(c,0)) for a,cs in SYN.items()}

def variant_cds(v, codons=None):
    s=list(wt_cds)
    for k,(aa,pos) in enumerate(zip(v,MUT_POSITIONS)):
        if PARD3[pos-1]==aa: continue
        c=(codons or PREF)[aa] if not isinstance(codons,dict) or aa in (codons or {}) else PREF[aa]
        s[(pos-1)*3:(pos-1)*3+3]=list(c)
    return "".join(s)

assert variant_cds("".join(PARD3[p-1] for p in MUT_POSITIONS))==wt_cds
specific=(w3>=0.8)&(w2<=0.2); promisc=(w3>=0.8)&(w2>=0.6)
mask=specific|promisc; lab=specific[mask]
print("discrimination set:", int(mask.sum()), f"({int(specific.sum())} specific, {int(promisc.sum())} promiscuous)")

def _rank(x):
    x=np.asarray(x,float); o=np.argsort(x); r=np.empty(len(x)); r[o]=np.arange(len(x),dtype=float)
    _,inv,cnt=np.unique(x,return_inverse=True,return_counts=True)
    m=np.zeros(len(cnt)); np.add.at(m,inv,r); m/=cnt; return m[inv]
def spearman(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float); ok=np.isfinite(a)&np.isfinite(b)
    return float(np.corrcoef(_rank(a[ok]),_rank(b[ok]))[0,1]) if ok.sum()>2 else np.nan
def auc(sc,lb):
    sc,lb=np.asarray(sc,float),np.asarray(lb,bool); p,n=sc[lb],sc[~lb]
    if not len(p) or not len(n): return np.nan
    r=_rank(np.concatenate([p,n]))
    return float((r[:len(p)].sum()-len(p)*(len(p)-1)/2)/(len(p)*len(n)))
def boot_ci(sc,lb,n=2000,seed=0):
    rng=np.random.default_rng(seed); sc,lb=np.asarray(sc,float),np.asarray(lb,bool); v=[]
    for _ in range(n):
        i=rng.integers(0,len(sc),len(sc))
        if lb[i].sum() in (0,len(i)): continue
        v.append(auc(sc[i],lb[i]))
    return (np.percentile(v,2.5),np.percentile(v,97.5)) if v else (np.nan,np.nan)

wt_code="".join(PARD3[p-1] for p in MUT_POSITIONS)
nmut=np.array([sum(a!=b for a,b in zip(v,wt_code)) for v in variants],float)
print(f"trivial baseline (mutation count) AUC = {auc(-nmut[mask], lab):.3f}")

## 5. ParD3 arms — this is the headline comparison

Three arms, matched to sections 14 and 19. The reference numbers to beat are the
trivial mutation-count baseline at **0.664** and the existing genomic models,
which sit between **0.39 and 0.60** — i.e. at chance.

The context dose-response is the important one. For Nucleotide Transformer, adding
real genomic flanks made things *worse* (0.505 at 0 nt down to 0.365 at 3 kb,
trend rho -0.771). Evo 2 is built for long context, so if the earlier failure was
about context rather than about the task, this is where it shows.

In [ ]:
import csv, time
rows=[]; t0=time.time()

def arm(name, seqs, note=""):
    s=evo2_loglik(seqs)
    a=auc(s[mask],lab); lo,hi=boot_ci(s[mask],lab)
    r=dict(arm=name, model=MODEL, auc=a, auc_lo=lo, auc_hi=hi,
           rho_on=spearman(s,w3), rho_off=spearman(s,w2),
           rho_margin=spearman(s,w3-w2), note=note)
    rows.append(r)
    print(f"{name:44s} AUC {a:.3f} [{lo:.3f},{hi:.3f}]  "
          f"rho_on {r['rho_on']:+.3f}  rho_off {r['rho_of']:+.3f}", flush=True)
    np.save(f"{OUTDIR}/evo2_scores_{name.replace(' ','_').replace('/','-')}.npy", s)
    return s

print("--- partner-blind: the CDS alone ---", flush=True)
arm("partner-blind (CDS alone)", [variant_cds(v) for v in variants])

print("\n--- partner-aware: the REAL operon (ParD3 and ParE3 overlap by 11 nt) ---", flush=True)
op, off = D["operon"], D["operon_offset"]
a3 = arm("partner-aware (REAL ParD3:ParE3 operon)",
         [op[:off] + variant_cds(v) + op[off+len(wt_cds):] for v in variants])
nc = D["noncognate"]
a2 = arm("partner-aware (ParD3:ParE2, synthetic)",
         [variant_cds(v) + nc[len(wt_cds):] for v in variants])
s = a3 - a2
print(f"{'partner-aware MARGIN (E3 - E2)':44s} AUC {auc(s[mask],lab):.3f}")
rows.append(dict(arm="partner-aware MARGIN (E3 - E2)", model=MODEL,
                 auc=auc(s[mask],lab), rho_margin=spearman(s,w3-w2)))

print("\n--- context dose-response on the real chromosome ---", flush=True)
win, woff = D["genomic_window"], D["genomic_cds_offset"]
for f in [0, 300, 1200, 3000, 5400]:
    lo_ = max(woff-f, 0); lo_ += (woff-lo_) % 6
    hi_ = min(woff+len(wt_cds)+f, len(win))
    pre, post = win[lo_:woff], win[woff+len(wt_cds):hi_]
    arm(f"context flank {f} nt", [pre+variant_cds(v)+post for v in variants],
        note=f"total {len(pre)+len(wt_cds)+len(post)} nt")

with open(f"{OUTDIR}/evo2_pard3.csv","w",newline="") as fh:
    k=sorted({x for r in rows for x in r}); w=csv.DictWriter(fh,fieldnames=k)
    w.writeheader(); w.writerows(rows)
print(f"\nwrote {OUTDIR}/evo2_pard3.csv  ({time.time()-t0:.0f}s)")

## 6. Synonymous floor

Synonymous encodings translate identically, so they carry the *same* measured
fitness by construction. Any score spread across them is provably
specificity-irrelevant, and its ratio to the between-variant spread bounds how
well any genomic proxy could possibly correlate. For Nucleotide Transformer this
ratio was ~0.5 with an attenuation ceiling of 0.83-0.89, so codon noise was
never sufficient to explain the null. This checks whether Evo 2 is cleaner.

In [ ]:
rng=np.random.default_rng(0)
idx=rng.choice(np.where(mask)[0], size=60, replace=False)
weights={a: np.array([usage.get(c,0) for c in SYN[a]],float) for a in SYN}
weights={a:(w/w.sum() if w.sum()>0 else np.ones(len(w))/len(w)) for a,w in weights.items()}

def sample_cds(v, rng):
    s=list(wt_cds)
    for aa,pos in zip(v,MUT_POSITIONS):
        if PARD3[pos-1]==aa: continue
        cs=SYN[aa]; c=cs[rng.choice(len(cs), p=weights[aa])]
        s[(pos-1)*3:(pos-1)*3+3]=list(c)
    return "".join(s)

within=[]
for vi in idx:
    seen=set()
    for _ in range(60):
        seen.add(sample_cds(variants[vi], rng))
        if len(seen)>=8: break
    if len(seen)<2: continue
    sc_=evo2_loglik(sorted(seen), progress=None)
    within.append(np.std(sc_,ddof=1))
between=evo2_loglik([variant_cds(variants[vi]) for vi in idx], progress=None)
w_sd=float(np.mean(within)); b_sd=float(np.std(between,ddof=1))
sig=float(np.sqrt(max(b_sd**2-w_sd**2,0))); atten=sig/b_sd if b_sd>0 else float("nan")
print(f"variants {len(within)}")
print(f"  within-variant SD (label-irrelevant): {w_sd:.3f}")
print(f"  between-variant SD                  : {b_sd:.3f}")
print(f"  ratio                               : {w_sd/b_sd:.3f}")
print(f"  attenuation ceiling on any rho      : {atten:.3f}")
import csv
with open(f"{OUTDIR}/evo2_synonymous_floor.csv","w",newline="") as fh:
    w=csv.writer(fh); w.writerow(["model","n","within_sd","between_sd","ratio","attenuation"])
    w.writerow([MODEL,len(within),w_sd,b_sd,w_sd/b_sd,atten])
print("wrote {OUTDIR}/evo2_synonymous_floor.csv")

## 7. The 25 DMS assays

Regenerated from public sources so nothing has to be uploaded: assay CSVs from the
ProteinGym release on HuggingFace, coding sequences from UniProt and ENA, each
accepted only if it translates to a protein *containing* the assayed target
exactly. Set `MAX_VARIANTS_PER_ASSAY` lower to trim runtime.

Reference numbers from the local run: ESM-2 650M mean rho **+0.466** (positive on
25/25), Nucleotide Transformer **-0.013**.

In [ ]:
MAX_VARIANTS_PER_ASSAY = 1500
import io, json, urllib.request, csv, time

PG="https://huggingface.co/datasets/OATML-Markslab/ProteinGym_v0.1/resolve/main"
def get(u, tries=3):
    for k in range(tries):
        try:
            with urllib.request.urlopen(u, timeout=120) as r: return r.read().decode()
        except Exception:
            if k==tries-1: raise
            time.sleep(2*(k+1))

def translate(d): return "".join(CODON.get(d[i:i+3],"X") for i in range(0,len(d)-2,3))

def embl_pids(acc):
    j=json.loads(get(f"https://rest.uniprot.org/uniprotkb/{acc}.json")); out=[]
    for x in j.get("uniProtKBCrossReferences",[]):
        if x.get("database")=="EMBL":
            for p in x.get("properties",[]):
                if p.get("key") in ("ProteinId","protein_sequence_id") and p.get("value") not in ("-",None):
                    out.append(p["value"])
    return out

def ena_cds(pid):
    for u in (f"https://www.ebi.ac.uk/ena/browser/api/fasta/{pid}",
              f"https://www.ebi.ac.uk/ena/browser/api/fasta/{pid.split('.')[0]}"):
        try: t=get(u,tries=1)
        except Exception: continue
        s="".join(l.strip() for l in t.splitlines() if l and not l.startswith(">"))
        if s: return s.upper()

ref=list(csv.DictReader(io.StringIO(get(f"{PG}/reference_files/DMS_substitutions.csv"))))
print(len(ref),"assays in the ProteinGym reference")

results=[]
RESOLVED_CDS=[]   # kept for the reading-frame test below
for r in ref:
    dms, acc, tgt = r["DMS_id"], r["UniProt_ID"], r["target_seq"]
    if not (60 <= len(tgt) <= 420): continue
    try: pids=embl_pids(acc)
    except Exception: continue
    hit=None
    for pid in pids[:12]:
        c=ena_cds(pid)
        if not c or len(c)%3: continue
        off=translate(c).rstrip("*").find(tgt)
        if off>=0: hit=(c,off); break
    if not hit: continue
    cdsf, off = hit
    region = cdsf[off*3:(off+len(tgt))*3]
    if translate(region)!=tgt: continue
    try: body=get(f"{PG}/DMS_ProteinGym_substitutions/{r['DMS_filename']}")
    except Exception: continue
    muts=[]
    for row in csv.DictReader(io.StringIO(body)):
        m=row.get("mutant","")
        if not m or ":" in m: continue
        try: p=int(m[1:-1]); s_=float(row["DMS_score"])
        except (ValueError,KeyError): continue
        if 1<=p<=len(tgt) and tgt[p-1]==m[0]: muts.append((p,m[0],m[-1],s_))
    if len(muts)<200: continue
    if len(muts)>MAX_VARIANTS_PER_ASSAY:
        sel=np.random.default_rng(0).choice(len(muts),MAX_VARIANTS_PER_ASSAY,replace=False)
        muts=[muts[i] for i in sel]
    seqs=[]
    for p,wt_,mu,_ in muts:
        s=list(region); s[(p-1)*3:(p-1)*3+3]=list(PREF[mu]); seqs.append("".join(s))
    RESOLVED_CDS.append((dms, region))
    sc_=evo2_loglik(seqs, progress=None)
    y=np.array([m[3] for m in muts]); rho=spearman(sc_,y)
    results.append(dict(dms_id=dms, n=len(y), target_len=len(tgt), rho_evo2=rho,
                        organism=r.get("source_organism","")))
    print(f"{dms[:44]:44s} n={len(y):5d}  rho_evo2 {rho:+.3f}", flush=True)
    with open(f"{OUTDIR}/evo2_dms.csv","w",newline="") as fh:
        k=sorted({x for q in results for x in q}); w=csv.DictWriter(fh,fieldnames=k)
        w.writeheader(); w.writerows(results)

if results:
    rr=np.array([q["rho_evo2"] for q in results]); ci=1.96*rr.std(ddof=1)/np.sqrt(len(rr))
    print(f"\n{len(results)} assays  mean rho {rr.mean():+.4f}  "
          f"95% CI [{rr.mean()-ci:+.4f}, {rr.mean()+ci:+.4f}]  positive {int((rr>0).sum())}/{len(rr)}")
    print("reference: ESM-2 650M +0.466 (25/25 positive), NT-v2 50M -0.013")

## 7b. The reading-frame test (CORRECTED DESIGN, read this before running)

**The earlier version of this cell was wrong and has been replaced.** It quoted
section 26 of `FINDINGS.md`, which was **retracted** on 31 Aug 2026. A rotation
(`cds[1:]+cds[:1]`) is, apart from one wrap junction, real genomic sequence read
one base later, and the reverse complement is real sequence on the other strand.
The two conditions that went unpenalised were exactly the two that yield real
genomic strings, so the whole result was reproduced by "is this a real genomic
string" with no appeal to reading frame.

**What replaces it.** A *matched* contrast: insert a stop codon's bases in frame,
and insert the same bases one nucleotide out of phase. Both are synthetic 3-base
edits to a real gene, so realness is matched and the confound cannot apply. Only
one of them creates a premature stop.

**How to read the number.** Effects are divided by each model's own reference
effect (mononucleotide shuffle), because nats per token are not comparable across
a 6-mer, a single-nucleotide and an amino-acid tokenizer. Calibration from a
3-periodic Markov model (the GeneMark device), which represents frame by
construction, against a matched aperiodic twin:

| model | contrast, as % of its own reference | |
|---|---|---|
| Markov order-2, 3-periodic | **+3.42%** [+2.63, +4.20] | positive control |
| Markov order-2, aperiodic | -0.54% [-1.28, +0.20] | negative control |
| ESM-2 35M | +4.88% [+1.47, +8.29] | detects |
| NT-v2 50M | +4.38% [-2.83, **+11.59**] | **underpowered, not null** |
| HyenaDNA 32k | +1.21% [-0.93, +3.35] | boundary tie |

**Read every null against 3.42%, never against zero.** At 24 unique coding
sequences NT-v2's interval reaches 11.59%, so a frame effect the size a
frame-representing model shows would not have been detected. Bounding it needs
roughly 214 sequences. Nothing currently supports the claim that a genomic model
lacks frame representation.

**So what would Evo 2 settle?** If Evo 2 clears 3.42% of its own reference with an
interval excluding it, that is a positive finding: frame representation at scale.
If its interval merely contains zero, report the interval and the implied minimum
detectable effect, and claim nothing. Given the power problem, a null here is
likely to be uninformative rather than meaningful, and saying so is the correct
outcome.

Sequences are deduplicated to unique coding sequences before any statistic is
computed, because the ProteinGym-derived set repeats proteins.

In [ ]:
import numpy as np, csv
from math import comb
COMPT = str.maketrans("ACGT", "TGCA")
rng2 = np.random.default_rng(0)
STOPS = ["TAA", "TAG", "TGA"]

def binom_p(k, n):
    return min(1.0, 2*sum(comb(n, i) for i in range(0, min(k, n-k)+1))/2**n)

def mono_shuffle(c):           # the reference effect: real vs base-composition noise
    return "".join(rng2.permutation(list(c)))

def stop_in_frame(c):
    """Replace one internal codon with a stop. Creates a premature stop."""
    n = len(c)//3
    i = int(rng2.integers(n//4, 3*n//4))
    return c[:i*3] + STOPS[int(rng2.integers(3))] + c[i*3+3:]

def stop_out_of_frame(c):
    """Same three bases, written one nucleotide off. Matched control: no stop."""
    n = len(c)//3
    i = int(rng2.integers(n//4, 3*n//4))
    p = i*3 + 1
    return c[:p] + STOPS[int(rng2.integers(3))] + c[p+3:]

# deduplicate: the assay set repeats proteins
seen, genes = set(), []
for name, s in [("ParD3_Lite_2020", wt_cds)] + list(RESOLVED_CDS):
    if len(s) % 3 or not (200 <= len(s) <= 2400) or set(s) - set("ACGT"):
        continue
    if s in seen:
        continue
    seen.add(s); genes.append((name, s))
print(f"{len(genes)} UNIQUE coding sequences")

def per_token(seqs):
    ll = evo2_loglik(seqs, progress=None)
    n = [max(len(model.tokenizer.tokenize(s)) - 1, 1) for s in seqs]
    return np.array(ll)/np.array(n, float)

real, ref, a_arm, b_arm = [], [], [], []
for i, (name, seq) in enumerate(genes, 1):
    v = per_token([seq, mono_shuffle(seq), stop_in_frame(seq), stop_out_of_frame(seq)])
    real.append(v[0]); ref.append(v[1]); a_arm.append(v[2]); b_arm.append(v[3])
    print(f"[{i:2d}/{len(genes)}] {name[:36]:36s} ref {v[0]-v[1]:+.4f}  "
          f"in-frame {v[0]-v[2]:+.4f}  out {v[0]-v[3]:+.4f}", flush=True)

real, ref, a_arm, b_arm = map(np.array, (real, ref, a_arm, b_arm))
d_ref = real - ref                 # reference effect
d = a_arm - b_arm                  # the matched contrast
n = len(d)
t = 2.069 if n == 24 else 1.96 + 0.5/max(n-2, 1)
se = d.std(ddof=1)/np.sqrt(n)
pct = 100*d.mean()/d_ref.mean()
lo = 100*(d.mean()-t*se)/d_ref.mean(); hi = 100*(d.mean()+t*se)/d_ref.mean()
k = int((d > 0).sum())

print(f"\n--- matched contrast, {n} unique sequences ---")
print(f"reference effect (mono shuffle) : {d_ref.mean():+.4f} nats/token")
print(f"contrast (in-frame minus out)   : {d.mean():+.4f}  in-frame lower {k}/{n}"
      f"  binom p={binom_p(k, n):.4f}")
print(f"as % of own reference           : {pct:+.2f}% [{lo:+.2f}%, {hi:+.2f}%]")
print(f"minimum detectable effect       : {100*t*se/d_ref.mean():.2f}% of reference")

print("\n=== VERDICT, against the 3.42% positive control ===")
if lo > 3.42:
    print(f"Evo 2 REPRESENTS reading frame: {pct:.2f}% of its own reference, interval\n"
          f"excludes the 3.42% a frame-aware model shows. Frame representation appears\n"
          f"at scale where NT-v2 could not be shown to have it.")
elif hi < 3.42:
    print(f"BOUNDED NULL: interval tops at {hi:.2f}%, below the 3.42% positive control,\n"
          f"so a frame effect that size is excluded. This IS informative.")
else:
    print(f"UNDERPOWERED, claim nothing. Interval [{lo:.2f}%, {hi:.2f}%] spans the 3.42%\n"
          f"positive control, so a real frame effect would not have been detected at\n"
          f"n={n}. Report the interval and the MDE. This is the same outcome NT-v2 gave\n"
          f"and it is not evidence either way.")

with open(f"{OUTDIR}/evo2_reading_frame.csv", "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["n_unique", "reference_effect", "contrast", "pct_of_reference",
                "ci_lo_pct", "ci_hi_pct", "in_frame_lower", "binom_p"])
    w.writerow([n, d_ref.mean(), d.mean(), pct, lo, hi, k, binom_p(k, n)])
print("\nwrote {OUTDIR}/evo2_reading_frame.csv")

## 8. Package the results

In [ ]:
import glob, zipfile, os
files=[p for p in glob.glob(f"{OUTDIR}/evo2_*.csv")+glob.glob(f"{OUTDIR}/evo2_scores_*.npy")]
with zipfile.ZipFile(f"{OUTDIR}/evo2_crosstalk_output.zip","w",zipfile.ZIP_DEFLATED) as z:
    for p in files: z.write(p, os.path.basename(p))
print("wrote {OUTDIR}/evo2_crosstalk_output.zip with", len(files), "files")
for p in sorted(files): print("  ", os.path.basename(p))
try:
    from google.colab import files as gf; gf.download(f"{OUTDIR}/evo2_crosstalk_output.zip")
except Exception as e:
    print("download it from the file browser on the left:", e)